<a href="https://colab.research.google.com/github/inaki-ruiz/Finance-and-Technology/blob/main/Tema_3_Optimization%20/src/cvxpy_intro_english.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.3 $\quad$ CVXPY

CVXPY is a domain-specific language for convex optimization in Python. Its main features include:

- **Natural Problem Formulation**:
  - It allows users to formulate convex optimization problems using high-level syntax that closely follows mathematical notation.

- **Compatibility with Well-Known Solvers**:
  - It provides access to widely used optimization solvers.

- **Convexity Verification**:
  - The library provides tools to check whether an optimization problem is convex.


There are essentially three approaches to solving convex optimization problems in Python:

- Implement existing optimization algorithms (solvers), such as Gurobi, SciPy, PuLP, GLPK, and CPLEX.

- Design your own optimization algorithm (e.g., using machine learning models).

- Use a high-level library (CVXPY) that integrates standard solvers through a high-level modeling language.


You can find the complete library documentation at [CVXPY](https://www.cvxpy.org/index.html).

Presentation: [Convex Optimization with CVXPY](https://www.youtube.com/watch?v=kXqu-TqEl7Q)


### Disciplined Convex Programming

#### Concept and Foundations
- **Definition**: Disciplined Convex Programming (DCP) is a system of rules and techniques for constructing convex optimization problems.
- **Objective**: Ensure that formulated optimization problems are convex, making it easier to find globally optimal solutions.

#### Key Principles
1. **Composition of Convex Functions**: Functions in a DCP problem must be combined according to rules that preserve convexity.
2. **Curvature Rules**: Determine whether an expression is convex, concave, or affine based on the operations and functions involved.
3. **DCP Constraints**: Only certain forms of constraints are allowed to preserve the problem's convexity.

#### Application in CVXPY
- **CVXPY and DCP**: CVXPY uses DCP to automatically check whether a formulated problem is convex.
- **Benefits**:
  - Simplifies the formulation of convex problems.
  - Helps prevent common errors by ensuring the problem is convex and therefore amenable to convex optimization methods.


### Components of an Optimization Problem Using CVXPY

Recall what we covered in the previous notebook:

Our goal is to **minimize/maximize $f_0(x| \theta)$** subject to $f_i(x| \theta) \leq b_i$, $\qquad i= 1,...,m$

where:

- $x = (x_1, x_2,..., x_n)$ are the variables.

- $\theta = (\theta_1, \theta_2,..., \theta_n)$ are the parameters.

- $f_0: \mathbb{R}^n \rightarrow \mathbb{R}$ is the objective function.

- $f_i: \mathbb{R}^n \rightarrow \mathbb{R}$ are the constraint functions.


In [1]:
import numpy as np
import pandas as pd
import cvxpy as cp

#### Decision Variables (Trainable Parameters)
- **Definition**: These represent the quantities we want to optimize, represented by the vector $w$.
- **Example**: In a resource allocation problem, the variables could represent the amount of resources allocated to different tasks.


To remember this, think of a `variable` as something whose value changes during the optimizer's search.


In [12]:
# A trainable variable with 2 components (2 weights)
w = cp.Variable(2)  # (w1, w2)
w

Variable((2,), var30)

In [13]:
# Equivalently, define a vector of variables directly (5 weights)
w = cp.Variable((5,))  # (w1, w2, w3, w4, w5)
w

Variable((5,), var31)

In [14]:
# We can also define a matrix of variables
m = cp.Variable((5, 2))
m
# [(m11, m12),
#  (m21, m22),
#  (m31, m32),
#  (m41, m42),
#  (m51, m52)]

Variable((5, 2), var32)

#### Objective Function
- **Definition**: The function we want to minimize or maximize, represented by $f_0$.
- **Example**: In a cost minimization problem, the objective function could be the total cost as a function of the decision variables.


In [15]:
# Minimize the sum of the components of vector w
cp.Minimize(cp.sum(w))

Minimize(Expression(AFFINE, UNKNOWN, ()))

In [16]:
# Minimize the L1 norm (sum of absolute values) of vector w
cp.Minimize(cp.norm(w, 1))

Minimize(Expression(CONVEX, NONNEGATIVE, ()))

In [17]:
# Maximize the sum of the logarithms of the components of vector w
cp.Maximize(cp.sum(cp.log(w)))

Maximize(Expression(CONCAVE, UNKNOWN, ()))

#### Constraints
- **Definition**: Conditions that the decision variables must satisfy, represented by $f_i$.
- **Example**: In a cost minimization problem, constraints could specify the minimum resources needed for each task.


- Constraints are modeled using equality and inequality expressions: `==`, `>=`, and `<=`.

- Strict inequalities `<` and `>` are not allowed. For example, $x < 5$ can be approximated by $x + \epsilon \leq 5$, where $\epsilon$ is a very small positive number.

- Inequality expressions are interpreted element by element, following NumPy-style rules for scalars, vectors, and matrices (broadcasting).


In [18]:
# Each of the 5 elements of vector w must be less than or equal to 6
w <= 6

Inequality(Variable((5,), var31))

In [19]:
# Each element of w must be at least the corresponding element of array c
c = np.array([1, 3, 5, 10, 2])
w >= c

Inequality(Constant(CONSTANT, NONNEGATIVE, (5,)))

In [20]:
# Expressions can be assigned to variables or grouped into lists
restriccion_1 = m <= 10
restriccion_2 = m >= 0
constraints = [restriccion_1, restriccion_2]
constraints.append(w <= c)
constraints

[Inequality(Variable((5, 2), var32)),
 Inequality(Constant(CONSTANT, ZERO, ())),
 Inequality(Variable((5,), var31))]

Because CVXPY builds on NumPy conventions, we can also define constraints on subsets of variables using indexing or slicing:


In [21]:
# The element at index [3, 1] of m must be <= 9
m[3, 1] <= 9

Inequality(Expression(AFFINE, UNKNOWN, ()))

In [22]:
# Elements in the first column of the first 4 rows must be <= 8
m[:4, 0] <= 8

Inequality(Expression(AFFINE, UNKNOWN, (4,)))

In [23]:
# We can combine ordinary Python code with constraint
# construction as needed

other_constraints = []
for i in range(w.shape[0]):
    other_constraints.append(w[i] >= i - 2)

other_constraints

[Inequality(Constant(CONSTANT, NONPOSITIVE, ())),
 Inequality(Constant(CONSTANT, NONPOSITIVE, ())),
 Inequality(Constant(CONSTANT, ZERO, ())),
 Inequality(Constant(CONSTANT, NONNEGATIVE, ())),
 Inequality(Constant(CONSTANT, NONNEGATIVE, ()))]

### Operators and Functions
The library treats the operators `+`, `-`, `*`, `/`, and `@` as functions, preserving NumPy's semantics.


In [24]:
w * 2

Expression(AFFINE, UNKNOWN, (5,))

In [25]:
w + w * 2

Expression(AFFINE, UNKNOWN, (5,))

In [26]:
# w has shape (5,)
# m has shape (5,2)
w @ m  # has shape (2,)

Expression(UNKNOWN, UNKNOWN, (2,))

In [27]:
# m has shape (5,2)
# m.T has shape (2,5)
m @ m.T

Expression(UNKNOWN, UNKNOWN, (5, 5))

Note: Some functions are applied element by element, just as in NumPy:


In [28]:
# Absolute value of each element of the vector
cp.abs(w)

Expression(CONVEX, NONNEGATIVE, (5,))

In [29]:
# Compute e^{m_ij} for each matrix element
cp.exp(m)

Expression(CONVEX, NONNEGATIVE, (5, 2))

The library provides a wide range of functions for performing most common mathematical operations.

- See the more complete list in the [documentation](https://www.cvxpy.org/tutorial/functions/index.html).



## Exercise: Optimal Allocation of Short-Term Financing

A company needs to raise **€100,000** to finance its short-term operations. It can borrow money from three financing sources:

| Financing source | Annual interest rate | Maximum available |
|---|---|---|
| Bank Loan A | 4% | €50,000 |
| Bank Loan B | 5% | €70,000 |
| Credit Line C | 7% | €40,000 |

The company wants to obtain the required financing while **minimizing its total annual interest cost**.

### Tasks

1. Define the decision variables representing the amount borrowed from each financing source.
2. Formulate the objective function to minimize the total annual interest cost.
3. Define the constraints:
   - The total amount borrowed must equal €100,000.
   - Borrowing from each source cannot exceed its maximum available amount.
   - Borrowing amounts must be non-negative.
4. Solve the optimization problem using **CVXPY**.
5. Display the optimal financing allocation and the minimum annual interest cost.


In [30]:
irr = np.array([0.04,0.05,0.07])
max_ava = [50000,70000,40000]


In [43]:
w = cp.Variable(3)

objective = cp.Minimize(irr @ w)

constrains = [cp.sum(w)== 100000,
              w <= max_ava,
              w >= 0]

problem = cp.Problem(objective,constrains)

problem.solve()

for a in w:
  print(a.value)

49999.99997716809
50000.000020724015
2.107896632341328e-06



## Exercise 2: Profit Maximization Under Budget Constraints

A company produces three products: A, B, and C. Each product generates a different contribution margin and requires an investment in working capital.

The company has a **working capital budget of €80,000** and limited production capacity.

| | Product A | Product B | Product C |
|---|---:|---:|---:|
| Contribution margin per unit (€) | 40 | 55 | 70 |
| Working capital per unit (€) | 20 | 35 | 50 |
| Maximum demand (units) | 1,500 | 1,200 | 1,000 |

The company can produce a maximum of **2,500 units** in total.

### Objective

Determine the optimal production quantities for each product to **maximize the total contribution margin**, subject to the company's financial and production constraints.

### Tasks

1. Define the decision variables representing the quantities produced of each product.
2. Formulate the objective function to maximize the total contribution margin.
3. Define the constraints:
   - Total working capital cannot exceed €80,000.
   - Total production cannot exceed 2,500 units.
   - Production of each product cannot exceed its maximum demand.
   - Production quantities must be non-negative.
4. Solve the optimization problem using **CVXPY**.
5. Display the optimal production quantities, maximum total contribution margin, and working capital used.

**Note:** For simplicity, treat production quantities as continuous variables.



In [46]:

w = cp.Variable(3)
margen = np.array([40, 55, 70])
capital_requerido = np.array([20, 35, 50])
demanda_maxima = np.array([1500, 1200, 1000])
objetivo = cp.Maximize(margen @ w)

restricciones = [
    cp.sum(capital_requerido @ w) <= 80000,
    cp.sum(w) <= 2500,
    w <= demanda_maxima,
    w >= 0]

problema = cp.Problem(objetivo, restricciones)
problema.solve()

print(f"Margen de contribución total máximo: €{problema.value:.2f}")
print(f"Unidades óptimas a producir de A: {w.value[0]:.2f}")
print(f"Unidades óptimas a producir de B: {w.value[1]:.2f}")
print(f"Unidades óptimas a producir de C: {w.value[2]:.2f}")

capital_usado = capital_requerido @ w.value
print(f"Capital de trabajo utilizado: €{capital_usado:.2f} de €80,000")
print(f"Producción total: {np.sum(w.value):.2f} unidades de 2,500")

Margen de contribución total máximo: €130000.00
Unidades óptimas a producir de A: 1108.32
Unidades óptimas a producir de B: 783.35
Unidades óptimas a producir de C: 608.32
Capital de trabajo utilizado: €80000.00 de €80,000
Producción total: 2500.00 unidades de 2,500
